- 실습 기본 환경 설정


In [ ]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

# 11-2 GAN: 적대적 경쟁으로 이미지를 생성하기

본 노트북은 본문 11-2절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- 역합성곱으로 이미지를 만들어 내는 DCGAN의 생성자와, 진짜/가짜를 판정하는 판별자
- DCGAN 논문이 권장하는 가중치 초기화
- 판별자와 생성자를 교대로 학습하는 손실 계산([그림 11-9])
- 무작위 잠재 벡터로 만든 이미지([그림 11-11])

## 데이터 준비

- 생성자의 마지막 활성화 함수가 `nn.Tanh`이므로 출력 범위가 `[-1, 1]`이다.
    - 진짜 이미지도 같은 범위에 있어야 판별자가 공정하게 비교할 수 있으므로, 평균 0.5, 표준편차 0.5로 정규화한다.

In [ ]:
# 참고 - Fashion-MNIST 를 [-1, 1] 범위로 정규화해 로드
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 데이터 변환 객체: 픽셀값을 [-1, 1] 범위로 정규화
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))      
])

data_root = '../../download'
# 훈련 데이터셋만 생성(DCGAN은 검증할 수 없음)
train_set = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
# 데이터로더 생성
BATCH_SIZE = 128
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'훈련 샘플 수: {len(train_set):,}')

## 생성자

- 잠재 벡터를 선형 계층으로 7x7 특징 지도로 바꾼 뒤, 두 번의 역합성곱(`nn.ConvTranspose2d`)으로 원본 이미지 크기까지 키운다.
    - 합성곱이 특징 지도를 줄이는 방향이라면, 역합성곱은 키우는 방향이다.
- 마지막 활성화 함수는 `nn.Tanh`로, 데이터의 정규화 범위와 맞춘다.

In [ ]:
######################################################################################
# 코드 11-7 - DCGAN의 생성자 클래스
######################################################################################

import torch.nn as nn

# 잠재 벡터로부터 가짜 이미지를 만드는 생성자 클래스
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.latent_dim = latent_dim
        # 잠재 벡터를 7x7 특징 지도로 변환
        self.fc_z_to_fmap = nn.Sequential(
            nn.Linear(latent_dim, 256 * 7 * 7),
            nn.BatchNorm1d(256 * 7 * 7),
            nn.ReLU() 
        )
        # 역합성곱으로 원본 크기로 복원: (256, 7, 7) -> (1, 28, 28)
        self.gen_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()                           # nn.Tanh의 출력 범위 [-1, 1]
        )

    def forward(self, z):
        h = self.fc_z_to_fmap(z)
        h = h.view(h.size(0), 256, 7, 7)
        return self.gen_conv(h)

## 판별자

- 이미지를 받아 진짜일 확률(0~1)을 출력한다.
    - 활성화 함수는 `nn.LeakyReLU(0.2)`, 마지막 활성화 함수는 `nn.Sigmoid`를 사용한다.
- `nn.LeakyReLU`는 음수 구간에서도 작은 기울기를 남겨, 판별자의 기울기가 죽는 것을 막는다.

In [ ]:
# 참고 - LeakyReLU 활성화 함수의 모양
import torch.nn.functional as F

# LeakyReLU: 음수 구간 기울기(negative_slope) 0.01 (기본값)
viz.plot_activation_compare(
    {'LeakyReLU (α=0.2)': lambda x: F.leaky_relu(x, negative_slope=0.2)},
    x_range=(-1, 1),
    figsize=(6, 4)
)

In [ ]:
######################################################################################
# 코드 11-8 - DCGAN의 판별자 클래스
######################################################################################

# 이미지가 진짜인지 가짜인지 판정하는 판별자 클래스
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()        
        self.disc_conv = nn.Sequential(     # (1, 28, 28) -> (64, 14, 14) -> (128, 7, 7)
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),              # LeakyReLU: 판별자에 사용(학습 안정화)
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),                   # 0과 1 사이의 값(진짜일 확률에 해당)
        )

    def forward(self, x):
        h = self.disc_conv(x)
        return self.fc(h)

## 가중치 초기화

- DCGAN 논문은 합성곱과 역합성곱의 가중치를 평균 0, 표준편차 0.02인 정규 분포로, 배치 정규화의 가중치를 평균 1, 표준편차 0.02로 초기화하도록 권장한다.
- 예제는 모델 클래스의 이름으로 계층 종류를 구분한다.
    - DCGAN 논문 구현에도 사용하는 방식인데, `Conv`라는 이름이 들어간 사용자 정의 클래스가 있으면 의도하지 않은 계층까지 초기화될 수 있으니 주의해야 한다.

In [ ]:
######################################################################################
# 코드 11-9 - DCGAN 모델의 가중치 초기화
######################################################################################

# 생성자와 판별자의 파라미터 초기화 함수
def weights_init(m):
    classname = m.__class__.__name__                # 클래스의 이름으로 계층 종류 구분
    if 'Conv' in classname:
        # 합성곱과 역합성곱: 평균 0, 표준편차 0.02의 정규 분포
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        # 배치 정규화: 평균 1.0의 정규 분포로 초기화, 처음에는 입력을 그대로 통과시킴
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


LATENT_DIM = 100        # 생성자 입력 잠재 벡터 크기

# 생성자와 판별자 객체 생성 및 가중치 초기화
D = Discriminator().to(device)
G = Generator(latent_dim=LATENT_DIM).to(device)   # LATENT_DIM: 잠재 벡터 크기(100)
# apply() 메서드: 모든 계층에 콜백 함수를 적용
G.apply(weights_init)                          
D.apply(weights_init)

common.print_param_summary({
    '생성자': G,
    '판별자': D
})

## 손실 함수와 옵티마이저

- 이진 교차 엔트로피 손실 함수(`nn.BCELoss`)와 Adam 옵티마이저를 사용한다.
    - DCGAN의 Adam은 `betas=(0.5, 0.999)`를 사용한다. 기본값보다 낮은 첫 번째 모멘텀 계수가 적대적 학습의 진동을 줄인다.
- 생성자와 판별자는 서로 다른 목표를 가지므로 옵티마이저도 따로 만든다.

In [ ]:
######################################################################################
# 코드 11-10 - DCGAN 모델의 학습에 사용하는 손실 함수와 옵티마이저
######################################################################################

import torch.optim as optim

# nn.BCELoss에서 집계 방법 인자(reduction)의 기본값: 평균(mean)
LR = 2e-4                               # 학습률: 안정적인 학습을 위해 작은 학습률 사용
criterion = nn.BCELoss()                # 이진 교차 엔트로피 손실 함수

# 생성자와 판별자를 따로 학습하므로 옵티마이저도 따로 사용
optim_g = optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
optim_d = optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

## 학습 함수

- 판별자와 생성자를 교대로 학습한다([그림 11-9]).
    - **판별자**: 진짜 이미지는 1로, 생성자가 만든 가짜 이미지는 0으로 판정하도록 학습한다. 두 손실을 합해 사용한다.
    - **생성자**: 판별자가 가짜 이미지를 1로 판정하도록 학습한다. 판별자를 속이는 것이 목표다.
- 판별자를 학습할 때는 생성자의 출력을 `detach()`해 생성자로 기울기가 흐르지 않게 한다.

In [ ]:
######################################################################################
# 코드 11-11 - DCGAN 모델의 학습 함수
######################################################################################

import torch

# DCGAN을 1 에포크 학습하는 학습 함수(G, D는 각각 생성자와 판별자 객체)
def train_epoch(G, D, train_loader, optim_g, optim_d, criterion, latent_dim, device):
    G.train()
    D.train()
    total_loss_d = 0.0
    total_loss_g = 0.0
    sample_size = 0

    # 데이터로더의 이미지 샘플은 모두 진짜이므로 레이블이 필요 없음
    for real_images, _ in train_loader:
        real_images = real_images.to(device)
        batch_size = real_images.size(0)
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)

        # ①판별자 학습: 진짜와 가짜 이미지를 모두 제대로 판별하도록 학습
        optim_d.zero_grad()
        # ①-a 진짜 이미지 판별 손실: 진짜 이미지(훈련 데이터)를 진짜로 판별했는가?
        loss_d_real = criterion(D(real_images), real_labels)
        # ①-b 가짜 이미지 판별 손실: 가짜 이미지(생성자가 생성)를 가짜로 판별했는가?
        # 임의의 잠재 벡터로부터 가짜 이미지 생성
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_images = G(z)
        # detach(): 판별자 학습 시 생성자 기울기가 함께 계산되지 않도록 분리
        loss_d_fake = criterion(D(fake_images.detach()), fake_labels)
        # 판별자 손실 합산
        loss_d = loss_d_real + loss_d_fake      
        # ①-c 기울기 계산 및 판별자 최적화: D.parameters()만 지정한 옵티마이저
        loss_d.backward()
        optim_d.step()

        # ②생성자 학습: 판별자가 속을 만큼 진짜 같은 가짜 이미지 생성 방법을 학습
        optim_g.zero_grad()
        z = torch.randn(batch_size, latent_dim).to(device)
        fake_images = G(z)                              # 가짜 이미지 생성
        # ②-a 생성자 손실: 가짜 이미지(생성자가 생성)로 판별자를 속였는가?
        loss_g = criterion(D(fake_images), real_labels)
        # ②-b 기울기 계산 및 생성자 최적화: G.parameters()만 지정한 옵티마이저
        loss_g.backward()
        optim_g.step()

        # 학습 로그 출력을 위한 손실 집계
        total_loss_d += loss_d.item() * batch_size
        total_loss_g += loss_g.item() * batch_size
        sample_size += batch_size

    # 샘플별 평균 손실 반환
    return total_loss_d / sample_size, total_loss_g / sample_size

- DCGAN은 검증 손실을 계산할 정답이 없으므로 훈련 손실만 추적한다. 본문과 같이 20 에포크를 학습한다.

In [ ]:
# 참고 - 전체 학습 루프 (검증 손실 없음, 본문 비공개 함수)
# 학습 로그·전체 학습 시간·손실 곡선은 공통코드컨벤션 §7.6/§7.7에 따라
# common.EpochLogger 로 통일한다. GAN 은 판별자/생성자 두 손실이 적대적으로
# 경쟁하므로 검증 손실이나 단조 감소하는 "최적 에포크" 개념이 약하다.
# minimize='판별자 손실'로 두지만 summary 의 최적 에포크는 참고용이며, 핵심은
# 전체 학습 시간과 두 손실 곡선이다(아래 log.plot 이 두 곡선을 함께 그린다).
EPOCHS = 20
log = common.EpochLogger(
    EPOCHS,
    columns=('판별자 손실', '생성자 손실'),
    formats=('{:.4f}', '{:.4f}'),
    minimize='판별자 손실',
    target_rows=EPOCHS
)
for epoch in range(1, EPOCHS + 1):
    loss_d, loss_g = train_epoch(
        G, D, train_loader,
        optim_g, optim_d, criterion, LATENT_DIM, device
    )
    log.row(epoch, loss_d, loss_g)
log.summary()

- 판별자와 생성자의 손실은 모두 한 배치에서 계산한 이진 교차 엔트로피 기반 값이라 한 그래프에 함께 그릴 수 있다([그림 11-10]).
    - 두 손실이 함께 줄어드는 것이 아니라 서로 밀고 당기며 진동하는 것이 정상이다.

In [ ]:
# 참고 - DCGAN 학습 곡선 시각화
# EpochLogger 가 보관한 두 손실 이력으로 판별자·생성자 곡선을 함께 그린다.
log.plot(title='DCGAN 학습 곡선', figsize=(6, 4))

## 이미지 생성([그림 11-11])

- 학습된 생성자에 무작위 잠재 벡터를 입력해 16장의 의류 이미지를 생성한다.
    - `nn.Tanh` 출력은 `[-1, 1]` 범위이므로 시각화하려면 0~1 범위로 변환해야 한다.

In [ ]:
######################################################################################
# 코드 11-12 - DCGAN의 이미지 생성
######################################################################################

# 무작위 잠재 벡터 16개 생성
z_test = torch.randn(16, LATENT_DIM).to(device)  
G.eval()
with torch.no_grad():
    generated = G(z_test)
    # nn.Tanh 출력은 [-1, 1] 범위이므로 시각화하려면 0~1 범위로 변환해야 함
    generated = (generated + 1) / 2

viz.plot_images(list(generated.cpu()), images_per_row=8)

- 입력 이미지 없이 무작위 잠재 벡터로 티셔츠, 바지, 신발 등 다양한 형태의 의류가 생성된다.
    - VAE 생성 결과와 달리 윤곽선이 선명하다.
- 다만 GAN은 특정 클래스의 분포가 아닌 학습 데이터 전체 분포를 학습하므로, 무작위 잠재 벡터로 어떤 이미지가 만들어질지 예측하기 어렵고 특정 클래스를 생성하도록 제어하기도 쉽지 않다.
    - 이 문제를 해결하기 위해 생성자와 판별자 양측에 클래스 레이블을 추가 조건으로 입력해 학습하는 조건부 GAN(cGAN)이 제안되었다.

## 정리

- GAN은 이미지를 만드는 생성자와 진짜/가짜를 판정하는 판별자가 경쟁하면서 함께 성장하는 구조다.
- DCGAN은 생성자에 역합성곱, 판별자에 합성곱을 사용하고, 가중치 초기화와 Adam 하이퍼파라미터까지 규약으로 정해 학습을 안정화했다.
- 판별자를 학습할 때는 생성자의 출력을 분리해 기울기가 섞이지 않게 해야 한다.
- GAN의 결과는 VAE보다 선명하지만, 생성 결과를 예측하거나 제어하기는 더 어렵다.